# 05 - RandomForest vs XGBoost
Same preprocessor (median + OneHot min_frequency=10). Trees need no scaling.
Society IS included - rare societies grouped, missing='Unknown'.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.train import load_data, build_models
from src.evaluate import regression_metrics
from sklearn.model_selection import train_test_split
import joblib
X, y, df = load_data('../data/raw/bengaluru_house_prices.csv')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = build_models()
for name in ['random_forest', 'xgboost']:
    pipe = models[name]
    pipe.fit(X_train, y_train)
    m = regression_metrics(y_test, pipe.predict(X_test))
    print(f"{name:13s} MAE={m['mae']:.2f} RMSE={m['rmse']:.2f} R2={m['r2']:.3f}")

In [ ]:
# save best + test one Whitefield house
from src.predict import predict_one
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(models['xgboost'], '../models/bengaluru_house_price_model.pkl')
print('saved ../models/bengaluru_house_price_model.pkl')
print('Whitefield 2BHK 1170sqft ->', round(predict_one('Super built-up  Area','Ready To Move','Whitefield','DuenaTa',1170,2,1,2, model_path='../models/bengaluru_house_price_model.pkl'),2), 'lakh')

In [ ]:
# feature importance (which columns drive XGB?)
import numpy as np
xgb = models['xgboost']
pre = xgb.named_steps['preprocessor']
names = pre.get_feature_names_out()
imp = xgb.named_steps['model'].feature_importances_
top = sorted(zip(names, imp), key=lambda t: -t[1])[:15]
for n, v in top:
    print(f'{v:.3f}  {n}')